# Phase 3 — Model Training

## Overview
This notebook trains all five models and logs every run to MLflow. The progression from simple to complex is intentional — each model must earn its complexity by outperforming the previous one.

| Model | Library | Why included |
|-------|---------|-------------|
| **Ridge Regression** | scikit-learn | Interpretable baseline; all complex models must beat this |
| **ARIMA/SARIMA** | pmdarima | Pure time-series benchmark; captures autocorrelation structure |
| **XGBoost** | xgboost + shap | Non-linear; typically best single model; SHAP-explainable |
| **LSTM** | TensorFlow/Keras | Sequence modelling; captures long-range temporal dependencies |
| **Ensemble (Stacking)** | scikit-learn / custom | Meta-learner combining all models; our headline result |

### Key design decisions
- **No data leakage**: `StandardScaler` is fit on training data only, applied to test
- **TimeSeriesSplit** for all cross-validation — never shuffle temporal data
- **MLflow** tracks every hyperparameter, metric, and model artifact for reproducibility

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
from src.features import load_processed, split, get_xy

df = load_processed()
train, test = split(df)
X_train, y_train = get_xy(train)
X_test,  y_test  = get_xy(test)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 1. Train All Models

`train_all()` runs the full training pipeline in sequence:
1. **Linear (Ridge)**: `RidgeCV` selects alpha from [0.01, 0.1, 1, 10, 100] via `TimeSeriesSplit(n_splits=5)`
2. **ARIMA**: `auto_arima` with stepwise search, AIC criterion, monthly seasonality (m=12)
3. **XGBoost**: `GridSearchCV` over `n_estimators`, `max_depth`, `learning_rate`, `subsample`, `colsample_bytree`; SHAP values computed and saved
4. **LSTM**: 2-layer LSTM(128→64), Dropout(0.2), EarlyStopping(patience=15); split-conformal prediction intervals computed on a held-out calibration set
5. **Ensemble (Stacking)**: Out-of-fold Ridge + XGBoost predictions → Ridge meta-learner

All models are serialized to `models/` and metrics are saved to `models/metrics.json`.

## 2. MLflow Experiment Log

Every training run is logged to the local MLflow tracking server in `mlruns/`. The table above shows:
- **Run name**: which model
- **MAE / RMSE / MAPE**: test-set error metrics
- **R²**: variance explained
- **Dir_Acc**: directional accuracy (% of months where model correctly predicted up/down)

To view interactively: `mlflow ui --backend-store-uri mlruns/` → open http://localhost:5000

## 3. Reload Predictions for Plotting

We reload the serialised bundles rather than relying on in-memory objects to confirm the saved models produce the same predictions. This is the same code path the API uses, so it validates the full serialization/deserialization round-trip.

## 4. Actual vs Predicted Plot

This is the primary visual result. Key things to look for:
- **Linear model** tracks the trend well but cannot capture sharp COVID spikes
- **ARIMA** reverts toward the mean too quickly after structural breaks
- **XGBoost** captures non-linear dynamics but may overfit locally
- **Ensemble** should show the smoothest and closest fit across the test period

A model that tracks 2021–2022 COVID inflation well is doing something genuinely difficult — most professional forecasters failed this period too.

In [ ]:
# Train all models — this may take a few minutes
from src.train import train_all
comparison = train_all(X_train, y_train, X_test, y_test)
print()
print(comparison)

In [ ]:
# View MLflow experiments
import mlflow
from pathlib import Path

mlflow.set_tracking_uri(str(Path('..') / 'mlruns'))
runs = mlflow.search_runs(experiment_names=['inflation-predictor'])
cols = ['tags.mlflow.runName', 'metrics.MAE', 'metrics.RMSE', 'metrics.MAPE', 'metrics.R2', 'metrics.Dir_Acc']
print(runs[cols].to_string())

In [ ]:
# Reload predictions for plotting
import joblib
from pathlib import Path

MODELS_DIR = Path('..') / 'models'

preds = {}

# Linear
b = joblib.load(MODELS_DIR / 'linear.joblib')
preds['Linear'] = b['model'].predict(b['scaler'].transform(X_test))

# XGBoost
b = joblib.load(MODELS_DIR / 'xgboost.joblib')
preds['XGBoost'] = b['model'].predict(X_test)

# ARIMA
b = joblib.load(MODELS_DIR / 'arima.joblib')
preds['ARIMA'] = b['model'].predict(n_periods=len(y_test))

In [ ]:
from src.evaluate import plot_predictions
plot_predictions(test.index, y_test.values, preds)